Sequential runner for ALL datasources.

Every value column / table is built as its own task, and the tasks run one after
another in this process. Each task writes to its own parquet file and, where a
temporary cache is used, its own isolated cache directory.


In [1]:
import os
import functools
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from datasources import (
    _month_compiler,
    _one_shot_compiler,
    _nemseer_pull,
    _dispatch_price,
    _dispatch_region_sum,
    _generation_fuel,
    _STTM_DWGM,
    _weather,
    _bid_availability,
    _bid_availability_fallback,
    _bid_prices,
    _bid_prices_fallback,
    _nem_registration_and_exemption_list,
)

try:
    os.chdir(Path(__file__).resolve().parent)
except NameError:
    os.chdir("/home/ec2-user/NEM-Short-Term-Price-Forecasting/EPF/1_Dataset")


In [2]:
_labels = []
_tasks = []

def _add(label, task):
    _labels.append(label)
    _tasks.append(task)


In [3]:
"""
Datasource 1

 - Datasource origin: nemosis
 - Datasource name: dispatch price     : AEMO's DISPATCHPRICE table — contains the Regional Reference Price (RRP) and related pricing outcomes published for each region at the end of every 5-minute dispatch interval
 - Variables:
    RRP         : Regional Reference Price ($/MWh) — the spot price for electricity in each NEM region, set by the market clearing process for each 5-minute dispatch interval
    REGIONID    : NEM region identifier used to filter and pivot the data (e.g. NSW1, QLD1, VIC1, SA1)
"""
_add("1_dispatch_price", functools.partial(
    _month_compiler,
    datasource_fetch_function = functools.partial(_dispatch_price, cache_dir="Pre_processing/temporary_cache/1"),
    start_date = pd.Timestamp("2018/01/01"),
    end_date = pd.Timestamp("2026/07/01"),
    datasource_file_path = Path("Processed_data/1_dispatch_price.parquet"),
    cache_dir = Path("Pre_processing/temporary_cache/1"),
))


"""
Datasource 2

 - Datasource origin: nemosis
 - Datasource name: dispatch region sum    : AEMO's DISPATCHREGIONSUM table — contains region-level demand, generation, and interconnector summary statistics published at the end of every 5-minute dispatch interval
 - Variables:
    TOTALDEMAND               : Total metered demand (MW) for the region in the dispatch interval
    AVAILABLEGENERATION       : Total available generation capacity (MW) that could be dispatched in the region
    NETINTERCHANGE            : Net interconnector flow (MW) into the region — positive means imports, negative means exports
    DEMANDFORECAST            : AEMO's forecast of regional demand (MW) for the dispatch interval
    DISPATCHABLEGENERATION    : Total generation (MW) actually dispatched in the region for the interval

    REGIONID                  : NEM region identifier used to filter and pivot the data (e.g. NSW1, QLD1, VIC1, SA1)
"""
_add("2_dispatch_region_sum", functools.partial(
    _month_compiler,
    datasource_fetch_function = functools.partial(_dispatch_region_sum, cache_dir="Pre_processing/temporary_cache/2"),
    start_date = pd.Timestamp("2018/01/01"),
    end_date = pd.Timestamp("2026/07/01"),
    datasource_file_path = Path("Processed_data/2_dispatch_region_sum.parquet"),
    cache_dir = Path("Pre_processing/temporary_cache/2"),
))


"""
Datasource 3
 - Datasource origin: nemosis
 - Datasource name: generation fuel   : Per-fuel-type aggregated generation (MW) from AEMO's DISPATCH_UNIT_SCADA table,
                                         with DUID-to-fuel mapping sourced from the NEM Registration and Exemption List.
 - Variables:
    coal_mw_{region}              : Total coal generation (MW)
    wind_mw_{region}              : Total wind generation (MW)
    solar_mw_{region}             : Total solar/PV generation (MW)
    hydro_mw_{region}             : Total hydro generation (MW)
    gas_mw_{region}               : Total gas/diesel/distillate generation (MW)
    battery_charge_mw_{region}    : Total battery charging (MW, positive)
    battery_discharge_mw_{region} : Total battery discharging (MW, positive)

 Note: requires Processed_data/0_nem_duid_mapping.parquet (see the optional
 _nem_registration_and_exemption_list() call in __main__ below).
"""
_add("3_generation_fuel", functools.partial(
    _month_compiler,
    datasource_fetch_function = functools.partial(_generation_fuel, cache_dir="Pre_processing/temporary_cache/3"),
    start_date = pd.Timestamp("2018/01/01"),
    end_date = pd.Timestamp("2026/07/01"),
    datasource_file_path = Path("Processed_data/3_generation_fuel.parquet"),
    cache_dir = Path("Pre_processing/temporary_cache/3"),
))


"""
Datasource 4

 - Datasource origin: www.aemo.com.au
 - Datasource name: STTM_DWGM
    STTM   : AEMO's Short Term Trading Market — daily ex ante commodity price ($/GJ) at each gas hub, set by the market clearing process for the upcoming gas day
    DWGM   : AEMO's Declared Wholesale Gas Market — 5-interval-per-day scheduled price ($/GJ) for Victoria; each interval covers a ~4-hour scheduling horizon (6am, 10am, 2pm, 6pm, 10pm)
 - Variables:
    gas_price_nsw : STTM Sydney hub ex ante daily commodity price ($/GJ)
    gas_price_qld : STTM Brisbane hub ex ante daily commodity price ($/GJ)
    gas_price_sa  : STTM Adelaide hub ex ante daily commodity price ($/GJ)
    gas_price_vic : DWGM Victoria scheduled price ($/GJ) — 5 intervals per gas day, forward-filled within each ~4-hour scheduling horizon
"""
_add("4_STTM_DWGM", functools.partial(
    _one_shot_compiler,
    datasource_fetch_function = _STTM_DWGM,
    start_date = pd.Timestamp("2018/01/01"),
    end_date = pd.Timestamp("2026/07/01"),
    datasource_file_path = Path("Processed_data/4_STTM_DWGM.parquet"),
))


"""
Datasource 5

 - Datasource origin: www.visualcrossing.com
 - Datasource name: weather
 - Variables:
    temp             : Mean air temperature (°C)
    feelslike        : Apparent temperature combining heat index and wind chill (°C)
    dew              : Dew point temperature — measure of atmospheric moisture (°C)
    humidity         : Relative humidity (%)
    precip           : Total precipitation (rain/snow liquid equivalent) for the period (mm)
    precipprob       : Probability of precipitation (%)
    preciptype       : Type(s) of precipitation (rain, snow, freezing rain, ice)
    snow             : New snowfall amount (cm)
    snowdepth        : Depth of snow currently on the ground (cm)
    windgust         : Maximum short-term wind gust speed (kph)
    windspeed        : Mean wind speed (kph)
    winddir          : Wind direction — degrees clockwise from north (°)
    sealevelpressure : Atmospheric pressure normalised to sea level (mb)
    cloudcover       : Fraction of sky covered by cloud (%)
    visibility       : Horizontal visibility distance (km)
    solarradiation   : Instantaneous solar radiation at the surface (W/m²)
    solarenergy      : Total solar energy accumulated over the period (MJ/m²)
    uvindex          : UV exposure index (0–10)
    severerisk       : Probability of severe weather events (%)
    conditions       : Short text summary of weather conditions
    icon             : Weather icon identifier
    stations         : Weather station(s) used as source for the observation
"""
_add("5_weather", functools.partial(
    _one_shot_compiler,
    datasource_fetch_function = _weather,
    start_date = pd.Timestamp("2018/01/01"),
    end_date = pd.Timestamp("2026/07/01"),
    datasource_file_path = Path("Processed_data/5_weather.parquet"),
))


"""
Datasource 6.1
 - Datasource origin: nemseer
 - Datasource name: predispatch price  : AEMO's PREDISPATCH PRICE table — contains 30-minute pre-dispatch pricing forecasts for each NEM region, published every half hour
 - Variables:
    RRP : Regional Reference Price forecast ($/MWh)
"""
_add("6_1_predispatch_price", functools.partial(
    _nemseer_pull,
    start_date = pd.Timestamp("2018/01/01"),
    end_date = pd.Timestamp("2026/07/01"),
    datasource_file_path = Path("Processed_data/6_1_predispatch_price.parquet"),
    nemseer_forecast_type = "PREDISPATCH",
    nemseer_table_name = "PRICE",
    value_cols = ["RRP"],
    run_col = "PREDISPATCH_RUN_DATETIME",
    interval_col = "DATETIME",
    entity_col = "REGIONID",
    cache_dir = Path("Pre_processing/temporary_cache/6_1"),
))


"""
Datasource 6.2
 - Datasource origin: nemseer
 - Datasource name: predispatch region sum  : AEMO's PREDISPATCH REGIONSUM table — contains 30-minute pre-dispatch demand and generation forecasts for each NEM region, published every half hour
 - Variables:
    TOTALDEMAND         : Forecast total regional demand (MW)
    AVAILABLEGENERATION : Forecast available generation capacity (MW)
    NETINTERCHANGE      : Forecast net interconnector flow (MW)
    DEMANDFORECAST      : AEMO's internal demand forecast (MW)
    AVAILABLELOAD       : Forecast available scheduled load (MW)

 All value columns share the same REGIONSUM table, so they are pulled in a
 single task: the table is downloaded once per month and every value column is
 pivoted into the one output parquet (instead of re-downloading it per column).
"""
_add("6_2_predispatch_regionsum", functools.partial(
    _nemseer_pull,
    start_date = pd.Timestamp("2018/01/01"),
    end_date = pd.Timestamp("2026/07/01"),
    datasource_file_path = Path("Processed_data/6_2_predispatch_regionsum.parquet"),
    nemseer_forecast_type = "PREDISPATCH",
    nemseer_table_name = "REGIONSUM",
    value_cols = ["TOTALDEMAND", "AVAILABLEGENERATION", "NETINTERCHANGE", "DEMANDFORECAST", "AVAILABLELOAD"],
    run_col = "PREDISPATCH_RUN_DATETIME",
    interval_col = "DATETIME",
    entity_col = "REGIONID",
    cache_dir = Path("Pre_processing/temporary_cache/6_2"),
))


"""
Datasource 6.3
 - Datasource origin: nemseer
 - Datasource name: predispatch interconnector solution  : AEMO's PREDISPATCH INTERCONNECTORRES table — contains 30-minute pre-dispatch MW flow forecasts for each NEM interconnector, published every half hour
 - Variables:
    MWFLOW        : Forecast MW flow on each interconnector
    METEREDMWFLOW : Metered MW flow forecast

 Both value columns share the same INTERCONNECTORRES table, so they are pulled
 in a single task (one download per month, both columns pivoted into one file).
"""
_add("6_3_predispatch_interconnectorsoln", functools.partial(
    _nemseer_pull,
    start_date = pd.Timestamp("2018/01/01"),
    end_date = pd.Timestamp("2026/07/01"),
    datasource_file_path = Path("Processed_data/6_3_predispatch_interconnectorsoln.parquet"),
    nemseer_forecast_type = "PREDISPATCH",
    nemseer_table_name = "INTERCONNECTORRES",
    value_cols = ["MWFLOW", "METEREDMWFLOW"],
    run_col = "PREDISPATCH_RUN_DATETIME",
    interval_col = "DATETIME",
    entity_col = "INTERCONNECTORID",
    cache_dir = Path("Pre_processing/temporary_cache/6_3"),
))


"""
Datasource 7
 - Datasource origin: nemseer
 - Datasource name: pdpasa region solution  : AEMO's PDPASA REGIONSOLUTION table — contains 30-minute demand and reserve forecasts for each NEM region, published every half hour
 - Variables:
    DEMAND10                  : 10th percentile demand forecast (MW)
    DEMAND50                  : 50th percentile demand forecast (MW)
    DEMAND90                  : 90th percentile demand forecast (MW)
    RESERVEREQ                : Reserve requirement (MW)
    SURPLUSRESERVE            : Surplus reserve above requirement (MW)
    MAXSURPLUSRESERVE         : Maximum surplus reserve (MW)
    MAXSPARECAPACITY          : Maximum spare capacity headroom (MW)
    AGGREGATEPASAAVAILABILITY : Sum of PASA availability across all scheduled generators + semi-scheduled UIGF (MW)

 All value columns share the same REGIONSOLUTION table, so they are pulled in a
 single task (one download per month, every column pivoted into one file).
"""
_add("7_1_pdpasa_regionsolution", functools.partial(
    _nemseer_pull,
    start_date = pd.Timestamp("2018/01/01"),
    end_date = pd.Timestamp("2026/07/01"),
    datasource_file_path = Path("Processed_data/7_1_pdpasa_regionsolution.parquet"),
    nemseer_forecast_type = "PDPASA",
    nemseer_table_name = "REGIONSOLUTION",
    value_cols = ["DEMAND10", "DEMAND50", "DEMAND90", "RESERVEREQ", "SURPLUSRESERVE",
                  "MAXSURPLUSRESERVE", "MAXSPARECAPACITY", "AGGREGATEPASAAVAILABILITY"],
    run_col = "RUN_DATETIME",
    interval_col = "INTERVAL_DATETIME",
    entity_col = "REGIONID",
    cache_dir = Path("Pre_processing/temporary_cache/7_1"),
))


"""
Datasource 8.1

 - Datasource origin: nemosis 
 - Datasource name: bid stack   : Per-DUID bid availability from AEMO's BIDPEROFFER_D table.
 - Variables (per DUID):
    {DUID}_maxavail : Total maximum available capacity (MW) for the dispatch interval
    {DUID}_bands    : Comma-separated string of the 10 band MW offers e.g. "22,0,150,0,..."
"""
_add("8_1_bid_availability", functools.partial(
    _month_compiler,
    datasource_fetch_function = functools.partial(_bid_availability, cache_dir="Pre_processing/temporary_cache/8_1"),
    fallback_fetch_function = functools.partial(_bid_availability_fallback, cache_dir="Pre_processing/temporary_cache/8_1"),
    start_date = pd.Timestamp("2018/01/01"),
    end_date = pd.Timestamp("2026/07/01"),
    datasource_file_path = Path("Processed_data/8_bid_availability.parquet"),
    cache_dir = Path("Pre_processing/temporary_cache/8_1"),
))


"""
Datasource 8.2

 - Datasource origin: nemosis 
 - Datasource name: bid prices  : Per-DUID daily price band offers from AEMO's BIDDAYOFFER_D table.
 - Variables (per DUID):
    {DUID}_prices : Comma-separated string of the 10 price band offers ($/MWh) e.g. "0,100,300,..."
"""
_add("8_2_bid_prices", functools.partial(
    _month_compiler,
    datasource_fetch_function = functools.partial(_bid_prices, cache_dir="Pre_processing/temporary_cache/8_2"),
    fallback_fetch_function = functools.partial(_bid_prices_fallback, cache_dir="Pre_processing/temporary_cache/8_2"),
    start_date = pd.Timestamp("2018/01/01"),
    end_date = pd.Timestamp("2026/07/01"),
    datasource_file_path = Path("Processed_data/8_bid_prices.parquet"),
    cache_dir = Path("Pre_processing/temporary_cache/8_2"),

))


In [4]:
if __name__ == "__main__":
    # Optional: (re)build the DUID→fuel mapping that Datasource 3 depends on.
    if False:
        _nem_registration_and_exemption_list()

    # Run every datasource one after another. Each task prints its own per-month
    # progress and writes its output parquet; failures are captured so one bad
    # datasource doesn't abort the rest of the run.
    results = []
    for label, task in tqdm(list(zip(_labels, _tasks)), desc="Datasources", unit="task"):
        try:
            task()
            results.append((label, None))
        except Exception:
            import traceback
            tb = traceback.format_exc()
            print(f"\n=== {label} FAILED ===\n{tb}", flush=True)
            results.append((label, tb))

    # Report any tasks that raised (run order matches _labels).
    failures = [(lbl, err) for lbl, err in results if err]
    if failures:
        print(f"\n{len(failures)} task(s) failed:", flush=True)
        for lbl, err in failures:
            print(f"\n=== {lbl} ===\n{err}", flush=True)
    else:
        print(f"\nAll {len(_tasks)} tasks completed successfully.", flush=True)


Datasources:   0%|          | 0/11 [00:00<?, ?task/s]

  Found 102 complete month(s) — will skip.
  1/102 2018-01 skipping (already processed).
  2/102 2018-02 skipping (already processed).
  3/102 2018-03 skipping (already processed).
  4/102 2018-04 skipping (already processed).
  5/102 2018-05 skipping (already processed).
  6/102 2018-06 skipping (already processed).
  7/102 2018-07 skipping (already processed).
  8/102 2018-08 skipping (already processed).
  9/102 2018-09 skipping (already processed).
 10/102 2018-10 skipping (already processed).
 11/102 2018-11 skipping (already processed).
 12/102 2018-12 skipping (already processed).
 13/102 2019-01 skipping (already processed).
 14/102 2019-02 skipping (already processed).
 15/102 2019-03 skipping (already processed).
 16/102 2019-04 skipping (already processed).
 17/102 2019-05 skipping (already processed).
 18/102 2019-06 skipping (already processed).
 19/102 2019-07 skipping (already processed).
 20/102 2019-08 skipping (already processed).
 21/102 2019-09 skipping (already pro

  Found 101 complete month(s) — will skip.
  1 incomplete month(s) will be re-fetched: ['2022-10']
  1/102 2018-01 skipping (already processed).
  2/102 2018-02 skipping (already processed).
  3/102 2018-03 skipping (already processed).
  4/102 2018-04 skipping (already processed).
  5/102 2018-05 skipping (already processed).
  6/102 2018-06 skipping (already processed).
  7/102 2018-07 skipping (already processed).
  8/102 2018-08 skipping (already processed).
  9/102 2018-09 skipping (already processed).
 10/102 2018-10 skipping (already processed).
 11/102 2018-11 skipping (already processed).
 12/102 2018-12 skipping (already processed).
 13/102 2019-01 skipping (already processed).
 14/102 2019-02 skipping (already processed).
 15/102 2019-03 skipping (already processed).
 16/102 2019-04 skipping (already processed).
 17/102 2019-05 skipping (already processed).
 18/102 2019-06 skipping (already processed).
 19/102 2019-07 skipping (already processed).
 20/102 2019-08 skipping (a